# Churn Prediction Pipeline
This notebook rewrites the original workflow into a repeatable pipeline that handles missing values, categorical encoding, class imbalance, cross-validation, and imbalanced evaluation metrics.

## Goals
- Clean and preprocess data safely
- Encode categorical variables without imposing arbitrary ordinality
- Use a stratified train/test split and cross-validation
- Handle class imbalance with SMOTE and class weighting
- Evaluate using recall, precision, F1, ROC-AUC, PR-AUC, and confusion matrix
- Provide a final inference-ready pipeline for deployment

In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from joblib import dump

warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

Libraries loaded successfully


In [4]:
# 2. Load the dataset
path = r"C:\Users\owner\Desktop\churn_prediction\data\Telco_Customer_Churn.csv"
df = pd.read_csv(path)

df = df.copy()  # Create a copy of the original DataFrame to avoid modifying it directly
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
# 3. Inspect data shape, types, and missing values
print('Shape:', df.shape)
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nTarget distribution:')
print(df['Churn'].value_counts(normalize=True).rename('ratio'))

Shape: (7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBi

In [6]:
# 4. Data cleaning
# Drop identifier because it does not carry predictive signal.
df = df.drop(columns=['customerID'])

# Convert TotalCharges to numeric; bad entries become NaN.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_total = df['TotalCharges'].isna().sum()
print(f'TotalCharges missing before imputation: {missing_total}')

# Median imputation for TotalCharges is robust to outliers.
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print('Missing values after imputation:')
print(df.isnull().sum())

TotalCharges missing before imputation: 11
Missing values after imputation:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [7]:
# 5. Confirm class imbalance and review numeric features
print('Churn value counts:')
print(df['Churn'].value_counts())
print('\nChurn rate: {:.2%}'.format(df['Churn'].value_counts(normalize=True).get('Yes', 0)))

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
print('\nNumeric summary:')
print(df[numeric_features].describe().T)

Churn value counts:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.54%

Numeric summary:
                 count         mean          std    min      25%       50%  \
tenure          7043.0    32.371149    24.559481   0.00    9.000    29.000   
MonthlyCharges  7043.0    64.761692    30.090047  18.25   35.500    70.350   
TotalCharges    7043.0  2281.916928  2265.270398  18.80  402.225  1397.475   

                    75%      max  
tenure            55.00    72.00  
MonthlyCharges    89.85   118.75  
TotalCharges    3786.60  8684.80  


## Preprocessing strategy
- Numeric features: median imputation and scaling
- Categorical features: one-hot encoding with `handle_unknown='ignore'`
- Build a pipeline so preprocessing is applied consistently during training and inference

In [9]:
# 6. Build preprocessing pipeline
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
category_features = [col for col in df.columns if col not in numeric_features + ['Churn']]

print('Numeric features:', numeric_features)
print('Categorical features:', category_features)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, category_features)
], remainder='drop', sparse_threshold=0)

Numeric features: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [11]:
# 7. Split data using stratified sampling to preserve class ratio
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Train positive ratio:', y_train.mean())
print('Test positive ratio:', y_test.mean())

Train shape: (4930, 19)
Test shape: (2113, 19)
Train positive ratio: 0.2653144016227181
Test positive ratio: 0.26549929010885


## Model and imbalance handling
We use a pipeline that applies preprocessing, SMOTE oversampling, and a class-weighted Random Forest. This keeps resampling inside the training pipeline and prevents test leakage.

In [ ]:
# 8. Create imbalanced-aware pipeline
rf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

param_distributions = {
    'clf__n_estimators': [100, 200, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

search.fit(X_train, y_train)
print('Best CV recall:', search.best_score_)
print('Best parameters:', search.best_params_)

In [ ]:
# 9. Evaluate the best model on the held-out test set
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('Balanced accuracy:', balanced_accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:', recall_score(y_test, y_pred))
print('F1 score:', f1_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_proba))
print('PR AUC:', average_precision_score(y_test, y_proba))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['No churn', 'Churn']))

In [ ]:
# 10. Plot confusion matrix and curves
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Test Confusion Matrix')
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC AUC = {roc_auc_score(y_test, y_proba):.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label=f'PR AUC = {average_precision_score(y_test, y_proba):.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

In [ ]:
# 11. Tune the probability threshold to maximize F1
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1])
best_idx = np.nanargmax(f1_scores)
best_threshold = thresholds[best_idx]

print('Best threshold for F1:', best_threshold)
print('F1 at best threshold:', f1_scores[best_idx])

threshold_preds = (y_proba >= best_threshold).astype(int)
print('Balanced accuracy:', balanced_accuracy_score(y_test, threshold_preds))
print('Precision:', precision_score(y_test, threshold_preds))
print('Recall:', recall_score(y_test, threshold_preds))
print('F1 score:', f1_score(y_test, threshold_preds))

In [ ]:
# 12. Save the final pipeline for inference
dump(best_model, '../model/churn_pipeline.joblib')
print('Saved trained pipeline to ../model/churn_pipeline.joblib')

## Summary
- Cleaned the dataset and imputed missing `TotalCharges` values.
- Built a preprocessing pipeline with numeric scaling and one-hot encoding.
- Used stratified train/test split and cross-validation for robust evaluation.
- Handled class imbalance using SMOTE and class_weight.
- Evaluated with balanced accuracy, precision, recall, F1, ROC-AUC, and PR-AUC.
- Saved the final inference-ready pipeline.